# Question 4 - 80. Remove Duplicates from Sorted Array II

Given an integer array `nums` sorted in **non-decreasing order**, remove some duplicates **in-place** such that each unique element appears **at most twice**. The **relative order** of the elements should be kept the **same**.

Since it is impossible to change the length of the array in some languages, you must instead have the result be placed in the **first part** of the array `nums`. More formally, if there are `k` elements after removing the duplicates, then the first `k` elements of `nums` should hold the final result. It does not matter what you leave beyond the first `k` elements.

Return `k` *after placing the final result in the first* `k` *slots of* `nums`.

Do **not** allocate extra space for another array. You must do this by **modifying the input array in-place** with O(1) extra memory.

**Custom Judge:**

    int[] nums = [...]; // Input array
    int[] expectedNums = [...]; // The expected answer with correct length

    int k = removeDuplicates(nums); // Calls your implementation

    assert k == expectedNums.length;
    for (int i = 0; i < k; i++) {
        assert nums[i] == expectedNums[i];
    }

If all assertions pass, then your solution will be accepted.

**Example 1:**

    Input: nums = [1,1,1,2,2,3]
    Output: 5, nums = [1,1,2,2,3,_]
    Explanation: Your function should return k = 5, with the first five elements of nums being 1, 1, 2, 2 and 3 respectively.
    It does not matter what you leave beyond the returned k (hence they are underscores).

**Example 2:**

    Input: nums = [0,0,1,1,1,1,2,3,3]
    Output: 7, nums = [0,0,1,1,2,3,3,_,_]
    Explanation: Your function should return k = 7, with the first seven elements of nums being 0, 0, 1, 1, 2, 3 and 3 respectively.
    It does not matter what you leave beyond the returned k (hence they are underscores).

**Constraints:**

- `1 <= nums.length <= 3 * 10^4`
- `-10^4 <= nums[i] <= 10^4`
- `nums` is sorted in **non-decreasing** order.

---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** Two pointers (reader / writer), comparing with the value two slots back

📘 **Revise first:** [Pattern A · read/write pointers](./0-concepts-array-and-string.ipynb) in the concepts notebook

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force (pop extra copies) | O(n²) | O(1) |
| 2️⃣ Optimized (compare with nums[k−2]) | O(n) | O(1) |

> 🧠 = plain-English intuition (how you'd explain it to a friend) · 🗣️ = **interview script**: what to say out loud at each step.

### 🧠 The problem in plain English

Same setup as Question 3: a **sorted** list, and we rewrite it in place. The rule is looser this time: each value may appear **up to two times**. A third, fourth, … copy must go.

```
[1, 1, 1, 2, 2, 3]   →   [1, 1, 2, 2, 3, _]   k = 5
       ↑ third 1 removed
```

**Everyday picture:** a shop has a rule of **"maximum 2 per customer"**. Customers arrive in a sorted line (all the Alices together, then all the Bobs…). You're the cashier writing receipts. Before writing one, you glance at your receipt pad: **if the receipt two entries above is already this same customer**, this customer has had their two, so skip them. Otherwise, write the receipt.

Why "two entries above"? Because the line is sorted. If the entry two above is Alice, the one between must also be Alice (nothing can sit between two Alices in sorted order), so Alice already has 2.

### Step 0 · Clarify before coding

🗣️ *"So it's the same as removing duplicates, except each value can stay up to two times. It's sorted, it's in place with O(1) extra memory, I keep the original order, and I return k with the answer in the first k slots."*

| Question to ask | Answer | Why it matters |
|---|---|---|
| Sorted? | **Yes**, non-decreasing. | Copies sit together, so I only need to look back a fixed distance. |
| In place, O(1) extra? | **Yes.** | No helper array or Counter. |
| Arrays of length ≤ 2? | Already valid. | Everything is kept, so return n. |
| Long runs like `[5,5,5,5,5]`? | Keep 2 → `[5,5]`. | A good edge case to test. |

### Step 1 · Brute force: the "obvious" way

**Plain-English idea:** walk through the list. If the current value is the same as the one **two positions earlier**, it's a third (or later) copy, so delete it with `pop`.

🗣️ *"The naive way is to scan and delete any element that equals the element two positions before it, since that means it's at least the third copy."*

**Why is it slow?** `pop(i)` from the middle of a list makes **every later element shift left by one** to close the gap (see the concepts notebook, Part 3.5). With a long run like 30,000 identical values, we'd delete ~30,000 times and shift up to ~30,000 elements each time. That's about **n² operations**.

🗣️ *"Each pop from the middle is O(n) because of the shifting, so this is O(n²) in the worst case."*

In [1]:
class SolutionBrute:
    def removeDuplicates(self, nums: list[int]) -> int:
        i = 2
        while i < len(nums):
            if nums[i] == nums[i - 2]:
                nums.pop(i)         # O(n) shift; don't advance i, a new value slid into i
            else:
                i += 1
        return len(nums)

### Step 2 · Optimize: the "aha" moment

> 💡 **Aha:** same fix as Questions 2 and 3: **don't delete, copy keepers forward** with a reader and a writer. The only new part is the **test for "is this a keeper?"**

- **Writer `k`**: where the next kept value goes. The kept section is `nums[0 : k]`.
- **Reader**: walks every value `x`.

**The keeper test:** *"Does the kept section already end with two copies of `x`?"*

Because everything is sorted, the kept section ends with two `x`s **exactly when `nums[k − 2] == x`**. If the value two slots back equals `x`, the slot between them must also be `x`.

So: **keep `x` if `k < 2` (the first two always fit) or `x != nums[k − 2]`.**

🗣️ *"I'll reuse the reader/writer idea. The write pointer k marks the end of the section I'm keeping. A new value x is allowed unless that section already ends in two copies of x, and because the array is sorted, that's exactly when nums[k − 2] equals x. The first two elements always pass. So for each x, if k is less than 2 or x differs from nums[k − 2], I write it at k and increment. One pass, O(n) time and O(1) space."*

#### 🤔 Why this approach, and why not the others?

| Option | Verdict | Plain-English reason |
|---|---|---|
| `pop` the extra copies | ❌ | Each pop shifts the tail, so O(n²). |
| Count with a dict/Counter, rebuild | ⚠️ | O(n) time, but needs extra memory. Breaks the O(1) rule. |
| Track "current value" and a "how many so far" counter | ✅ | Also O(n)/O(1) and very readable, but there's more state to manage (reset the counter when the value changes). |
| **Compare with `nums[k − 2]`** | ✅ **best** | One condition, no extra counters, and it generalises: "at most t copies" means compare with `nums[k − t]`. |

**Why `nums[k − 2]` and not `nums[r − 2]` (two back in the *original* reading position)?** Because the reader may be far ahead of the writer. The value at `r − 2` might be a copy we **skipped**, and positions behind the writer may have been **overwritten**. The kept section `nums[:k]` is the only reliable record of what we've actually kept.

In [2]:
class Solution:
    def removeDuplicates(self, nums: list[int]) -> int:
        k = 0                                   # writer: kept section is nums[:k]
        for x in nums:                          # reader
            if k < 2 or x != nums[k - 2]:       # kept section doesn't already end in x, x
                nums[k] = x
                k += 1
        return k

### Step 3 · Toy example walkthrough (tell it like a story)

`nums = [1, 1, 1, 2, 2, 3]`

| reading x | k before | nums[k−2] | keep? | action | kept section nums[:k] |
|---|---|---|---|---|---|
| 1 | 0 | n/a | ✅ (k < 2) | nums[0] = 1 | [1] |
| 1 | 1 | n/a | ✅ (k < 2) | nums[1] = 1 | [1, 1] |
| 1 | 2 | nums[0] = **1** | ❌ 1 == 1 | skip (third 1) | [1, 1] |
| 2 | 2 | nums[0] = 1 | ✅ 2 ≠ 1 | nums[2] = 2 | [1, 1, 2] |
| 2 | 3 | nums[1] = 1 | ✅ 2 ≠ 1 | nums[3] = 2 | [1, 1, 2, 2] |
| 3 | 4 | nums[2] = 2 | ✅ 3 ≠ 2 | nums[4] = 3 | [1, 1, 2, 2, 3] |

Return **k = 5** ✅

**Narrated:** "The first two 1s go straight in. For the third 1, I look two slots back in what I've kept and see a 1, so I already have two 1s and skip it. The first 2: two back is a 1, so it's new and I keep it. The second 2: two back is still a 1, so I keep it too. Then the 3: two back is a 2, so keep. I've kept 5 values."

In [3]:
def run(cls, nums):
    nums = nums[:]
    k = cls().removeDuplicates(nums)
    return k, nums[:k]

cases = [
    ([1, 1, 1, 2, 2, 3], (5, [1, 1, 2, 2, 3])),
    ([0, 0, 1, 1, 1, 1, 2, 3, 3], (7, [0, 0, 1, 1, 2, 3, 3])),
    ([1], (1, [1])),
    ([2, 2], (2, [2, 2])),
    ([5, 5, 5, 5, 5], (2, [5, 5])),        # long run
    ([1, 2, 3], (3, [1, 2, 3])),
]
for nums, expected in cases:
    assert run(SolutionBrute, nums) == expected
    assert run(Solution, nums) == expected
    print(nums, "->", expected)

import random
for _ in range(500):
    nums = sorted(random.randint(0, 4) for _ in range(random.randint(1, 12)))
    assert run(Solution, nums) == run(SolutionBrute, nums)
print("All tests passed ✅")

[1, 1, 1, 2, 2, 3] -> (5, [1, 1, 2, 2, 3])
[0, 0, 1, 1, 1, 1, 2, 3, 3] -> (7, [0, 0, 1, 1, 2, 3, 3])
[1] -> (1, [1])
[2, 2] -> (2, [2, 2])
[5, 5, 5, 5, 5] -> (2, [5, 5])
[1, 2, 3] -> (3, [1, 2, 3])
All tests passed ✅


### Step 4 · Complexity in plain words, and wrap up

| | Time | Extra memory | In plain words |
|---|---|---|---|
| Pop extra copies | O(n²) | O(1) | Every delete shifts the rest of the list. |
| **Compare with `nums[k−2]`** | **O(n)** | **O(1)** | Read each value once; write each keeper once. |

**Why O(n) time?** One loop over the n values, with one comparison and at most one write per value.

**Why O(1) memory?** Only the writer `k` (and the loop variable) is stored.

**Edge cases to mention:** length 1 or 2, a very long run of one value, no duplicates at all.

**The general version (a great thing to volunteer):** to allow **at most `t` copies**, use `if k < t or x != nums[k − t]`. With `t = 1` this is exactly Question 3.

---

#### 🗣️ Full interview script (about 60 seconds)

*"The array is sorted and each value can appear at most twice; I need to compact it in place and return the new length.*

*The naive approach deletes any element equal to the one two positions earlier, but each deletion shifts the rest of the array, so that's O(n²).*

*Instead I use a write pointer k for the section I'm keeping. For each value x, I ask whether the kept section already ends with two copies of x. Since the array is sorted, that's true exactly when nums[k − 2] equals x. So if k is less than 2 or x differs from nums[k − 2], I write x at k and increment k. I compare against the kept section, not the original positions, because the original positions may have been skipped or overwritten.*

*One pass, O(n) time, O(1) extra space. And it generalises: for at most t copies, compare with nums[k − t]."*